# FCS analysis fof UFA2 from Accurate FRET II (Agam et. al Nat Meth 2023)

Load relevant modules

In [1]:
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

import smfbursts as smf

import smfbursts.fcs as fcs

Since the data is split over several files, we load each separetly with the load_put command,
and create a list at the end to make it easier to apply the same operation to all of them.

In [2]:
raw1 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_1.ptu')
raw4 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_4.ptu')
raw5 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_5.ptu')
raw6 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_6.ptu')
raw7 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_7.ptu')
raw8 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_8.ptu')
raw9 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_9.ptu')
raw10 = smf.loadraw.load_ptu('Lab8_U2AF2/mystery_protein_2_holo_ulm_10.ptu')
raws = [raw1, raw4, raw5, raw6, raw7, raw8, raw9, raw10]

FileNotFoundError: [Errno 2] No such file or directory: 'Lab8_U2AF2/mystery_protein_2_holo_ulm_1.ptu'

Display the setup group, fields with `None` need to be filled out.

In [ ]:
raw1.setup

Plot the altenation histogram of 1 of the files to choose where to assign the excitation windows

In [ ]:
smf.plot.alternation_hist(raw1)

Fill out in a loop all missing fields, first in setup, then in photon_data assign detectors and excitation periods

In [ ]:
for r in raws:
    # missing setup fields
    r.setup['num_spectral_ch'] = 2
    r.setup['num_polarization_ch'] = 2
    r.setup['num_split_ch'] = 1
    r.setup['excitation_wavelengths'] = np.array([532e-9, 642e-9])
    r.setup['detection_wavelengths'] = np.array([585e-9, 698e-9])
    r.setup['excitation_cw'] = np.array([False, False])
    r.setup['excitation_alternated'] = np.array([False, False])
    r.setup['detectors']['label'] = np.array(['ATTO 488', 'ATTO 643'])

    # remove the spectral_ploarization_split because it is there to identify detectors, but is not part of HDF5 spec
    r.photon_data[0].meas_specs['detectors_specs'].pop('spectral_polarization_split_chN', None)
    # 
    r.photon_data[0].meas_specs['detectors_specs']['spectral_ch1'] = np.array([2,3], dtype=np.uint8)
    r.photon_data[0].meas_specs['detectors_specs']['spectral_ch2'] = np.array([4,5], dtype=np.uint8)
    r.photon_data[0].meas_specs['detectors_specs']['polarization_ch1'] = np.array([2,4], dtype=np.uint8)
    r.photon_data[0].meas_specs['detectors_specs']['polarization_ch2'] = np.array([3,5], dtype=np.uint8)
    r.photon_data[0].meas_specs['alex_excitation_period1'] = np.array([1850,3000])
    r.photon_data[0].meas_specs['alex_excitation_period2'] = np.array([70,1500])

Plot alternation hist again to verify correctly assigned excitation periods

In [ ]:
smf.plot.alternation_hist(raw1)

We first make a list of individual data objects, then put into PhotonDataList 
so we can easily do the same pySMFS operation to all files.

In [ ]:
datas = [smf.photonHDF5.regularize_dets(r) for r in raws]
data = smf.PhotonDataList(datas)

Plot time trace to check data quality

In [ ]:
smf.plot.timetrace(datas[0], streams=smf.fretfactory.ALEXdefaults['streams'][1:], 
                   direction=[True, False, False], tmin=1, tmax=2,
                   stream_kwargs=smf.fretfactory.ALEXdefaults['stream_colors'][1:])

## Calculation of basic correlations

Correlations based on streams can now be calculated.

`fcs.correlate` is the base function for simple all-data correlations.

As the first line of the cell bellow shows, it has defaults for most arguments
such that the autocorrelation can be calculated simply by supplying a `PhotonData` or `PhotonDataList` data object.
The bin size is set such that there are 5 $\Delta\tau$ logrithmicly spaced bins per order of magnitude, from the clock rate to 1 second.

The `streamT` and `streamU` arguments identify which photons to correlate 
with `streamT` as the "start" time of the correlation and `streamU` as the "stop" time.
By default if only `streamT` is specified, it assumes an auto-correlation is being computed, and assigns that stream to `streamU`.
If both are specified and are different, cross-correlation is computed.

In [ ]:
corrlAll, binsAll = fcs.correlate(data) # correlation of all
corrlDex, binsDex = fcs.correlate(data, streamT=smf.PhSel('0ex'), bins=2) # correlation of just Dex streams
corrlAex, binsAex = fcs.correlate(data, streamT=smf.PhSel('1ex1em'), bins=np.logspace(-6,0,31)) # correlation of Acceptor stream
corrlCrossDA, binsCrossDA = fcs.correlate(data, streamT=smf.PhSel('0ex'), streamU=smf.PhSel('1ex1em')) # cross correlate
corrlCrossAD, binsCrossAD = fcs.correlate(data, streamT=smf.PhSel('1ex1em'), streamU=smf.PhSel('0ex')) # reverser cross correlation

# plot correlations
plt.stairs(corrlAll, binsAll, label='auto all photons')
plt.stairs(corrlDex, binsDex, label='auto Dex', color='g')
plt.stairs(corrlAex, binsAex, label='auto Aex', color='purple')
plt.stairs(corrlCrossDA, binsCrossDA, label='cross Dex X Aex', color='r')
plt.stairs(corrlCrossAD, binsCrossAD, label='cross Aex X Dex', color='m')
plt.xscale('log')
plt.legend()

## Burst Analysis for Purified FCS

Next we will demonstrate purified FCS, for which a basic burst search is needed.
First background parameters need to be defined:

In [ ]:
bg = smf.fretfactory.make_bg(data, func=smf.bg.exp_mlefit, period=60.0, auto_threshold=True, F_bg=2.0)

Examine the background

In [ ]:
smf.plot.time_plot(data.datas[0], bg['BgDD'], marker='o')
smf.plot.time_plot(data.datas[0], bg['BgDA'], marker='o')
smf.plot.time_plot(data.datas[0], bg['BgAA'], marker='o')
smf.plot.time_plot(data.datas[0], bg['BgAll'], marker='o')
plt.xlim(0)
plt.legend()

Define burst search parameters

In [ ]:
bs = smf.fretfactory.make_burst_search(bg=bg['bg'], m=10, F=6.0, streams=(smf.PhSel('0ex_1em')))

Now since we want to perform FCS on the FRET active species, we need to plot the E-S plot and determine gate thresholds to isolate the FRET population.

In [ ]:
g = smf.make_geq_gate(bs['NphActive_bg'], 50) & smf.make_geq_gate(bs['NphDex_bg'], 5) & smf.make_geq_gate(bs['NphAA_bg'], 5)

fig = plt.figure(figsize=(8,8))

ax, *_ = smf.plot.jointplot(data, bs['E_bg'], bs['S_bg'], gate=g, 
                            hplot_kwargs={'bins':smf.fretfactory.ALEXdefaults['ratio_bins']},
                            cplot_kwargs={'point_func':smf.plot.density_kde, 'cmap':'Spectral_r', 's':0.5})
ax[0].set_xlim([-0.2, 1.2])
ax[0].set_ylim([-0.2, 1.2])
ax[0].axvline(0.2)
ax[0].axhline(0.3)
ax[0].axhline(0.75)

Make a "FRET" gate for purified FCS

In [ ]:
gFret = g & smf.make_geq_gate(bs['E_bg'], 0.2) & smf.make_geq_gate(bs['S_bg'], 0.3) & smf.make_lt_gate(bs['S_bg'], 0.75)

### Compute purified FCS

The `fcs.purified_fcs()` function performs auto-correlation of only bursts [Laurence et. al. 2007](https://doi.org/10.1529/biophysj.106.093591).
This requires that a `Param` that defines the bursts be supplied as well.
All keyword arguments from the `fcs.correlate` function apply to `fcs.purified_fcs()`.
Usually with purified FCS, the bursts are "expanded" by some amount before and after the ending, this can be controled by the `expand` keyword argument, the units are seconds.

In [ ]:
corrlAll, binsAll = fcs.purified_fcs(data, bs['bursts'].regate(gFret))
corrlCross, binsCross = fcs.purified_fcs(data, bs['bursts'], gate=gFret, streamT=smf.PhSel('0ex'), streamU=smf.PhSel('1ex1em'), expand=0.01)
corrlCrossF, binsCrossF = fcs.purified_fcs(data, bs['bursts'], gate=gFret, streamT=smf.PhSel('0ex0em'), streamU=smf.PhSel('0ex1em'))
plt.stairs(corrlAll, binsAll, label="auto all")
plt.stairs(corrlCross, binsCross, label="cross Dex X Aex")
plt.stairs(corrlCrossF, binsCrossF, label="cross DD X DA")
plt.xscale('log')
plt.legend()

## Flourescence Lifetime Correlation Spectroscopy / filtered FCS

A more powerful technique, though also more difficult, if photon data includes nanotimes, is Fluorescence lifetime Correlation Spectroscopy, 
where set of fitler functions allows photons to contribute differently to the correlations based on the nanotime.
These filter functions are based on the TCSPC decays of the fluorescent species.

For this, first, the decays must be determiend.
In this example, this is done by taking the extremes of the respective, interconverting subpopulations.

> **Note**
>
> In this example all filters are created from "real" decays, another aproach is to create
> artificial, and therefore noise-free decays based on exponential fits.

In [ ]:
smf.plot.hist_bar(data, bs['E_bg'], gate=gFret, bins=np.linspace(-0.2, 1.2, 61));
plt.axvline(0.3)
plt.axvline(0.9)

Define gates for extremes of fluorescent subpopulations.

In [ ]:
gLow = gFret & smf.make_lt_gate(bs['E_bg'], 0.3)
gHigh = gFret & smf.make_geq_gate(bs['E_bg'], 0.9)

Now we need to extract the nanotime histograms per state.
Note also that since this data includes polarization (MFD) the parallel and perpendicular streams must be
assigned separate chanels, they are concatenated in the output filter (concatenation will be done later)

## Load IRF

In some versions of fFCS/FLCS, filters for buffer/scatter and dark-counts are also included.
In this example, both these filters will be included.
The dark counts is simply a flat decay (same value in all tcspc bins), so that will be created later.
But to determine the IRF, we need to load an IRF, here based on the measure of buffer.
(If this example were based on fits, we would be creating gaussians)

In [ ]:
irf = smf.loadraw.load_ptu('Lab8_U2AF2/BSA+buffer.ptu')

As the buffer and protein samples were measured under the same microscope settings, we can apply the same values for both when loading the raw data.

In [ ]:
irf.setup = raw1.setup
irf.photon_data[0].meas_specs = raw1.photon_data[0].meas_specs
smf.plot.alternation_hist(irf)

In [ ]:
irf = smf.photonHDF5.regularize_dets(irf)

In [ ]:
nhirf = fcs.extract_decays(irf, smf.PhSel('0ex_1ex1em'), nanobin=16)
nhLow = fcs.extract_decays(data, smf.PhSel('0ex_1ex1em'), bursts=bs['bursts'], gate=gLow, nanobin=16)
nhHigh = fcs.extract_decays(data, smf.PhSel('0ex_1ex1em'), bursts=bs['bursts'], gate=gHigh, nanobin=16)

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.semilogy(nhirf/nhirf.sum(), label='IRF')
ax.semilogy(nhLow/nhLow.sum(), label='Low')
ax.semilogy(nhHigh/nhHigh.sum(), label='High')
ax.legend()

Plot the filters. Note that the `fcs.get_weights()` function is a convenience function.
It is useful to know how good the filters will be, but the next function also performs this computation based on the decay.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
wghts = fcs.get_weights(data, [np.ones(nhirf.shape), nhLow, nhHigh], streams=smf.PhSel('0ex_1ex1em'), nanobin=16)
for w in wghts:
    ax.plot(w)
ax.axhline(0.0, c='k')

### All Data FLCS

Computing the lifetime filtered FCS is as simple as providing the data, and a list of the decays.
The same set of kwargs work in `fcs.flcs()` as `fcs.correlate()`

In [ ]:
corr, bins = fcs.flcs(data, [np.ones(nhLow.size), nhLow, nhHigh], streams=smf.PhSel('0ex_1ex1em'), nanobin=16)
mbins = bins[:-1] + np.diff(bins)/2

In [ ]:
plt.scatter(mbins, corr[1,1], label='Low')
plt.scatter(mbins, corr[2,2], label='High')
plt.title('Auto Correlations All Data')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbins, corr[1,2], label='Low X High')
plt.scatter(mbins, corr[2,1], label='High X Low')
plt.xscale('log')
plt.title('Cross Correlations All Data')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbins, corr[0,0], label='Background')
plt.title('Auto Correlations All Data bg-like')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbins, corr[0,1], label='BG X Low')
plt.scatter(mbins, corr[1,0], label='Low X BG')
plt.scatter(mbins, corr[0,2], label='BG X High')
plt.scatter(mbins, corr[2,0], label='High X BG')
plt.xscale('log')
plt.legend()
plt.title('Cross correlations All Data bg X FRET (should be 0)')
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
corrb, binsb = fcs.flcs(data.datas[0], [np.ones(nhLow.size), nhLow, nhHigh], bursts=bs['bursts'].regate(gFret), streams=smf.PhSel('0ex_1ex1em'), nanobin=16)
mbinsb = binsb[:-1] + np.diff(binsb)/2

In [ ]:
plt.scatter(mbinsb, corrb[1,1], label='Low')
plt.scatter(mbinsb, corrb[2,2], label='High')
plt.title('Auto Correlations Burst Purified')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbinsb, corrb[1,2], label='Low X High')
plt.scatter(mbinsb, corrb[2,1], label='High X Low')
plt.xscale('log')
plt.title('Cross Correlations Burst Purified')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbinsb, corrb[0,0], label='Background')
plt.title('Auto Correlations Burst Purified bg-like')
plt.xscale('log')
plt.legend()
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

In [ ]:
plt.scatter(mbinsb, corrb[0,1], label='BG X Low')
plt.scatter(mbinsb, corrb[1,0], label='Low X BG')
plt.scatter(mbinsb, corrb[0,2], label='BG X High')
plt.scatter(mbinsb, corrb[2,0], label='High X BG')
plt.xscale('log')
plt.legend()
plt.title('Cross correlations Burst Purified bg X FRET (should be 0)')
plt.xlabel(r'$\Delta\tau\:(s)$')
plt.ylabel(r'$g(\Delta\tau)$')

### Experimental: Use H<sup>2</sup>MM to generate filters

In [ ]:
import burstH2MM as bhm

In [ ]:
def dwell_dict(statepath:smf.Param, update=None, **kwargs)->dict[str:smf.Param|smf.Column]:
    update = dict() if update is None else update
    update['dwell'] = smf.Param(bhm.Dwells, statepath=statepath)
    update['Bstates'] = smf.Column(statepath, 'bstates')
    update['State'] = smf.Column(update['dwell'], 'state')
    smf.fretfactory.make_fret_from_base(update['dwell'], update=update, **kwargs)
    return update

In [ ]:
bsdc = smf.fretfactory.make_burst_search(bg=bg['bg'], m=10, F=6.0, streams=(smf.PhSel('0ex'), smf.PhSel('1ex1em')))
gdcAll = smf.make_geq_gate(bsdc['NphDex_bg'], 100) & smf.make_geq_gate(bsdc['NphAA_bg'], 50)

In [ ]:
bsdc['statepaths'] = bhm.StatePath.optimize_models(data, bsdc['bursts'].regate(gdcAll), 
                                                   streams=(smf.PhSel('0ex0em'), smf.PhSel('0ex1em'), smf.PhSel('1ex1em')))

bsdc['dwells'] = [dwell_dict(sp, skip=['S_raw', 'S_bg', 'NphAA_raw', 'NphAA_bg', 'NphDactive_raw', 'NphDactive_bg']) 
                  for sp in bsdc['statepaths']]

fig, ax = plt.subplots(1,3,figsize=(10,4))
bhm.plot.scatter_ICL(data, bsdc['statepaths'], ax=ax[0])
bhm.plot.scatter_pathBIC(data, bsdc['statepaths'], ax=ax[1])
bhm.plot.scatter_BIC(data, bsdc['statepaths'], ax=ax[2])

In [ ]:
n = 3
smf.plot.scatter(data, bsdc['E_raw'], bsdc['S_raw'], gate=gdcAll, s=1.0, alpha=0.5, point_func=smf.plot.density_kde)
bhm.plot.scatter_model(bsdc['E_raw'], bsdc['S_raw'], statepath=bsdc['statepaths'][n], data=data.datas[0], c='r')
# bhm.plot.scatter_model_trans_arrows(bs['E_raw'], bs['S_raw'], bs['statepaths'][n], data)
plt.xlim([0,1])
plt.ylim([0,1])

In [ ]:
bhm.StatePath.get_model_value(bsdc['E_raw'], bsdc['statepaths'][n]), bhm.StatePath.get_model_value(bsdc['S_raw'], bsdc['statepaths'][n])

In [ ]:
nhHmm = list()
for i in range(n+1):
    gstate = smf.make_isin_gate(bsdc['dwells'][n]['State'], i)
    nhHmm.append(fcs.extract_decays(data, smf.PhSel('0ex_1ex1em'), bursts=bsdc['dwells'][n]['dwell'], gate=gstate, nanobin=16))
    plt.semilogy(nhHmm[-1], label=f'state {i}')
plt.legend()

In [ ]:
weights = fcs.get_weights(data, nhHmm, streams=smf.PhSel('0ex0em'), nanobin=16)
for i, w in enumerate(weights):
    plt.plot(w, label=f'state {i}')
plt.legend()

In [ ]:
corrl, bins = fcs.flcs(data, [np.ones(nhHmm[1].size),]+ nhHmm[1::2], streams=smf.PhSel('0ex0em'), nanobin=16)
corrlb, binsb = fcs.flcs(data, nhHmm, streams=smf.PhSel('0ex0em'), bursts=bs['bursts'].regate(gFret), nanobin=16)

In [ ]:
mbins = bins[:-1] + np.diff(bins)/2
mbinsb = binsb[:-1] + np.diff(binsb)/2

In [ ]:
plt.scatter(mbins[2:], -corrl[1,2][2:], label='High X Low')
plt.scatter(mbins[2:], -corrl[2,1][2:], label='Low X High')
plt.xscale('log')
plt.legend()

In [ ]:
plt.scatter(mbins[2:], corrl[1,1][2:], label='High X High')
plt.scatter(mbins[2:], corrl[2,2][2:], label='Low X Low')
plt.xscale('log')
plt.legend()

In [ ]:
weights = fcs.get_weights(data, [np.ones(nhHmm[1].shape)] + nhHmm[1::2], streams=smf.PhSel('0ex0em'), nanobin=16)
for i, w in enumerate(weights):
    plt.plot(w, label=f'state {i}')
plt.legend()

In [ ]:
corrl, bins = fcs.flcs(data, [np.ones(nhHmm[1].size)] + nhHmm[1::2], streams=smf.PhSel('0ex0em'), nanobin=16)
corrlb, binsb = fcs.flcs(data, nhHmm, streams=smf.PhSel('0ex0em'), bursts=bs['bursts'].regate(gFret), nanobin=16)

In [ ]:
plt.scatter(mbinsb, corrlb[1,1], label='High')
plt.scatter(mbinsb, corrlb[2,2], label='Low')

plt.scatter(mbinsb, -corrlb[1,2], label='High X Low')
plt.scatter(mbinsb, -corrlb[2,1], label='Low X High')
plt.xscale('log')
plt.legend()

In [ ]:
for c in smf.get_citations():
    print(c)